# 청년기 우울증 조기 탐지 프로젝트 Pandas 시각화
수업 실습 방식에 맞춰 CSV를 `pd.read_csv()`로 불러오고, `groupby`와 matplotlib을 사용해 인사이트별 시각화 자료를 생성한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 220
Path('figures').mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv('term_project_insight_data.csv')
print(df.shape)
df.head()

In [ ]:
# Reference high는 PHQ-9 합산 점수 10점 이상을 뜻한다.
display(df['reference_high'].value_counts())
display(df['reference_high'].value_counts(normalize=True) * 100)

In [ ]:
def high_rate(frame, group_col, labels):
    order = list(labels.values())
    temp = frame.dropna(subset=[group_col]).copy()
    temp['group_label'] = temp[group_col].map(labels)
    temp['group_label'] = pd.Categorical(temp['group_label'], categories=order, ordered=True)
    out = temp.groupby('group_label', sort=True, observed=False)['reference_high'].agg(count='count', rate='mean').reset_index()
    out['rate'] = out['rate'] * 100
    return out

def add_labels(ax):
    for p in ax.patches:
        h = p.get_height()
        ax.text(p.get_x() + p.get_width()/2, h + 0.18, f'{h:.2f}%', ha='center', va='bottom', fontsize=9)

def save_bar(data, title, path, color='#2f80ed'):
    fig, ax = plt.subplots(figsize=(8.2, 4.7))
    ax.bar(data['group_label'], data['rate'], color=color, edgecolor='#374151', linewidth=0.4)
    ax.set_title(title, fontsize=14, weight='bold', pad=12)
    ax.set_ylabel('Reference high 비율(%)')
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(0, data['rate'].max() * 1.25 + 0.3)
    add_labels(ax)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.show()

In [ ]:
# 그림 1: 고스트레스와 생활 맥락 결합 집단의 Reference high 비율
# 캡처용: 그래프 내부 제목 없음
stress_context_specs = [
    ('고스트레스', (df['stress_level'] == 0)),
    ('고스트레스+\n1인 가구', (df['stress_level'] == 0) & (df['single_person_household'] == 1)),
    ('고스트레스+\n비경제활동', (df['stress_level'] == 0) & (df['economic_activity'] == 1)),
    ('고스트레스+\n흡연 경험', (df['stress_level'] == 0) & (df['smoking_status'] == 0)),
    ('고스트레스+\n저활동', (df['stress_level'] == 0) & (df['walking_days'] == 0)),
]
stress_context_rate = pd.DataFrame({
    'group_label': [name for name, _ in stress_context_specs],
    'rate': [df.loc[mask, 'reference_high'].mean() * 100 for _, mask in stress_context_specs],
    'count': [int(mask.sum()) for _, mask in stress_context_specs],
})

fig, ax = plt.subplots(figsize=(8.8, 4.8))
colors = ['#9ca3af', '#6b7280', '#4b5563', '#374151', '#111827']
bars = ax.bar(
    stress_context_rate['group_label'],
    stress_context_rate['rate'],
    color=colors,
    edgecolor='#374151',
    linewidth=0.8,
)
ax.set_ylabel('Reference high 비율(%)')
ax.grid(axis='y', alpha=0.25)
ax.set_ylim(0, stress_context_rate['rate'].max() * 1.25 + 0.5)
for bar, rate in zip(bars, stress_context_rate['rate']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.25,
        f'{rate:.2f}%',
        ha='center',
        va='bottom',
        fontsize=10,
    )
fig.tight_layout()
fig.savefig('figures/01_stress_context_reference_high_notitle.png', bbox_inches='tight', pad_inches=0.08)
plt.show()


In [ ]:
# 그림 2: 위험 신호 개수에 따른 Reference high 비율
# 캡처용: 그래프 내부 제목 없음
risk_signal_specs = [
    ('고스트레스', df['stress_level'] == 0),
    ('1인 가구', df['single_person_household'] == 1),
    ('비경제활동', df['economic_activity'] == 1),
    ('흡연 경험', df['smoking_status'] == 0),
    ('저활동', df['walking_days'] == 0),
]
risk_signal_count = sum(mask.astype(int) for _, mask in risk_signal_specs)
risk_signal_rows = []
for signal_count in range(5):
    if signal_count < 4:
        mask = risk_signal_count == signal_count
        label = f'{signal_count}개'
    else:
        mask = risk_signal_count >= 4
        label = '4개 이상'
    risk_signal_rows.append({
        'group_label': label,
        'rate': df.loc[mask, 'reference_high'].mean() * 100,
        'count': int(mask.sum()),
    })
risk_signal_rate = pd.DataFrame(risk_signal_rows)

fig, ax = plt.subplots(figsize=(8.8, 4.8))
colors = ['#9ca3af', '#6b7280', '#4b5563', '#374151', '#111827']
bars = ax.bar(
    risk_signal_rate['group_label'],
    risk_signal_rate['rate'],
    color=colors,
    edgecolor='#374151',
    linewidth=0.8,
)
ax.set_ylabel('Reference high 비율(%)')
ax.grid(axis='y', alpha=0.25)
ax.set_ylim(0, risk_signal_rate['rate'].max() * 1.25 + 0.5)
for bar, rate, count in zip(bars, risk_signal_rate['rate'], risk_signal_rate['count']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.25,
        f'{rate:.2f}%\n(n={count:,})',
        ha='center',
        va='bottom',
        fontsize=10,
    )
fig.tight_layout()
fig.savefig('figures/02_risk_signal_count_reference_high_notitle.png', bbox_inches='tight', pad_inches=0.08)
plt.show()


In [ ]:
# 인사이트 1: 스트레스 수준과 고스트레스 결합 집단
stress_rate = high_rate(df, 'stress_level', {0.0:'고스트레스', 1.0:'저스트레스'})
combo_specs = [
    ('고스트레스+1인 가구', (df['stress_level']==0) & (df['single_person_household']==1)),
    ('고스트레스+비경제활동', (df['stress_level']==0) & (df['economic_activity']==1)),
    ('고스트레스+흡연 경험', (df['stress_level']==0) & (df['smoking_status']==0)),
    ('고스트레스+여성', (df['stress_level']==0) & (df['sex']==1)),
    ('고스트레스+저활동', (df['stress_level']==0) & (df['walking_days']==0)),
]
combo_rate = pd.DataFrame({
    'group_label': [name for name, _ in combo_specs],
    'rate': [df.loc[mask, 'reference_high'].mean()*100 for _, mask in combo_specs]
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={'width_ratios':[0.8, 1.6]})
axes[0].bar(stress_rate['group_label'], stress_rate['rate'], color=['#ef4444', '#60a5fa'], edgecolor='#374151', linewidth=0.4)
axes[0].set_title('스트레스 수준별 고위험군 비율', fontsize=11, weight='bold')
axes[0].set_ylabel('Reference high 비율(%)')
axes[0].grid(axis='y', alpha=0.25)
axes[0].set_ylim(0, 18)
add_labels(axes[0])
axes[1].bar(combo_rate['group_label'], combo_rate['rate'], color='#ef4444', edgecolor='#374151', linewidth=0.4)
axes[1].set_title('고스트레스와 생활 맥락이 결합된 집단', fontsize=11, weight='bold')
axes[1].set_ylabel('Reference high 비율(%)')
axes[1].grid(axis='y', alpha=0.25)
axes[1].set_ylim(0, 18)
axes[1].tick_params(axis='x', rotation=20)
add_labels(axes[1])
fig.suptitle('스트레스가 생활 조건과 결합될 때 높아지는 위험 신호', fontsize=14, weight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/01_stress_combo_reference_high.png', bbox_inches='tight')
plt.show()

In [ ]:
# 인사이트 2: 1인 가구와 혼인 상태
household = high_rate(df, 'single_person_household', {1.0:'1인 가구', 0.0:'2인 이상'})
marital = high_rate(df, 'marital_status', {1.0:'미혼·별거 등', 0.0:'기혼'})

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8))
for ax, title, data in zip(axes, ['가구 형태', '혼인 상태'], [household, marital]):
    ax.bar(data['group_label'], data['rate'], color='#2563eb', edgecolor='#374151', linewidth=0.4)
    ax.set_title(title, fontsize=11, weight='bold')
    ax.set_ylabel('Reference high 비율(%)')
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(0, data['rate'].max() * 1.25 + 0.3)
    add_labels(ax)
fig.suptitle('고립 가능성과 관련된 사회적 맥락별 고위험군 비율', fontsize=14, weight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/02_household_marital_reference_high.png', bbox_inches='tight')
plt.show()

In [ ]:
# 인사이트 3: 흡연, 걷기, 음주 등 생활 리듬
lifestyle = pd.concat([
    high_rate(df, 'smoking_status', {0.0:'흡연 경험 있음', 1.0:'흡연 경험 없음'}).assign(variable='흡연'),
    high_rate(df, 'walking_days', {0.0:'저활동', 1.0:'고활동'}).assign(variable='걷기'),
    high_rate(df, 'alcohol_frequency', {0.0:'고빈도 음주', 1.0:'저빈도 음주'}).assign(variable='음주'),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(9.2, 4.8))
colors = {'흡연':'#2563eb', '걷기':'#10b981', '음주':'#f59e0b'}
ax.bar(lifestyle['group_label'], lifestyle['rate'], color=[colors[v] for v in lifestyle['variable']], edgecolor='#374151', linewidth=0.4)
ax.set_title('생활습관·일상 리듬 관련 집단별 고위험군 비율', fontsize=14, weight='bold', pad=12)
ax.set_ylabel('Reference high 비율(%)')
ax.grid(axis='y', alpha=0.25)
ax.set_ylim(0, lifestyle['rate'].max() * 1.25 + 0.3)
ax.tick_params(axis='x', rotation=18)
add_labels(ax)
fig.tight_layout()
fig.savefig('figures/03_lifestyle_reference_high.png', bbox_inches='tight')
plt.show()

In [ ]:
# 인사이트 4: 경제·건강 취약성
vulnerability = pd.DataFrame({
    'group_label': ['기초생활수급 경험', '사고·중독 경험', '고혈압 진단 경험', '당뇨병 진단 경험'],
    'rate': [
        df.loc[df['basic_livelihood_recipient']==0, 'reference_high'].mean()*100,
        df.loc[df['accident_poisoning_experience']==0, 'reference_high'].mean()*100,
        df.loc[df['hypertension_diagnosis']==0, 'reference_high'].mean()*100,
        df.loc[df['diabetes_diagnosis']==0, 'reference_high'].mean()*100,
    ]
})
save_bar(vulnerability, '경제·건강 취약성 집단의 고위험군 비율', 'figures/04_vulnerability_reference_high.png', color='#f59e0b')

In [ ]:
# 인사이트 5: 성별·지역·연령대
sex = high_rate(df, 'sex', {1:'여성', 0:'남성'})
region = high_rate(df, 'region_group', {0:'수도권', 1:'비수도권'})
age = df.dropna(subset=['age_band']).groupby('age_band', sort=True, observed=False)['reference_high'].agg(count='count', rate='mean').reset_index()
age['rate'] = age['rate'] * 100
age = age.rename(columns={'age_band':'group_label'})

fig, axes = plt.subplots(1, 3, figsize=(12, 4.8), gridspec_kw={'width_ratios':[0.8, 0.9, 1.6]})
for ax, title, data, color in [
    (axes[0], '성별', sex, '#6366f1'),
    (axes[1], '지역', region, '#2563eb'),
    (axes[2], '연령대', age, '#10b981'),
]:
    ax.bar(data['group_label'].astype(str), data['rate'], color=color, edgecolor='#374151', linewidth=0.4)
    ax.set_title(title, fontsize=11, weight='bold')
    ax.set_ylabel('Reference high 비율(%)')
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(0, data['rate'].max() * 1.35 + 0.3)
    add_labels(ax)
fig.suptitle('성별·지역·연령대별 고위험군 비율', fontsize=14, weight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/05_demo_context_reference_high.png', bbox_inches='tight')
plt.show()

In [ ]:
# 벤치마크: 모델 성능지표 비교
metrics = pd.read_csv('benchmark_metrics.csv')
if 'selection' in metrics.columns:
    metrics = metrics[metrics['selection'].eq('fixed_decision_threshold')].copy()
elif 'threshold_strategy' in metrics.columns:
    metrics = metrics[metrics['threshold_strategy'].eq('fixed_decision_threshold')].copy()
metrics['model'] = pd.Categorical(metrics['model'], ['Logistic Regression', 'XGBoost', 'LightGBM', 'Random Forest'], ordered=True)
metrics = metrics.sort_values('model')
metric_cols = ['auroc', 'pr_auc', 'positive_recall', 'positive_precision', 'positive_f1', 'balanced_accuracy']
metric_labels = ['AUROC', 'PR-AUC', 'Recall', 'Precision', 'F1', 'Balanced Acc.']

fig, ax = plt.subplots(figsize=(10.8, 5.4))
x = np.arange(len(metrics))
width = 0.12
for i, (col, label) in enumerate(zip(metric_cols, metric_labels)):
    ax.bar(x + (-2.5+i)*width, metrics[col], width=width, label=label)
ax.set_title('Matplotlib 기반 모델 벤치마크 지표 비교', fontsize=14, weight='bold', pad=12)
ax.set_xticks(x, metrics['model'].astype(str), rotation=15)
ax.set_ylim(0, 0.9)
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.25)
ax.legend(ncol=3, fontsize=9)
fig.tight_layout()
fig.savefig('figures/06_benchmark_metrics_matplotlib.png', bbox_inches='tight')
plt.show()

In [ ]:
# 벤치마크: LightGBM 변수 중요도(SHAP)
shap = pd.read_csv('shap_variable_contribution.csv')
if 'mean_abs_shap' not in shap.columns:
    value_col = [c for c in shap.columns if c != 'feature'][0]
    shap = shap.rename(columns={value_col:'mean_abs_shap'})
top = shap.sort_values('mean_abs_shap', ascending=False).head(10).sort_values('mean_abs_shap')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.barh(top['feature'], top['mean_abs_shap'], color='#2563eb', edgecolor='#374151', linewidth=0.4)
ax.set_title('Matplotlib 기반 LightGBM 변수 중요도(SHAP)', fontsize=14, weight='bold', pad=12)
ax.set_xlabel('Mean absolute SHAP')
ax.grid(axis='x', alpha=0.25)
for p in ax.patches:
    w = p.get_width()
    ax.text(w + top['mean_abs_shap'].max()*0.015, p.get_y()+p.get_height()/2, f'{w:.3f}', va='center', fontsize=9)
fig.tight_layout()
fig.savefig('figures/07_shap_variable_contribution_matplotlib.png', bbox_inches='tight')
plt.show()